# Modeling Sentimen Berita Ekonomi CNBC Indonesia

Notebook ini berisi tahap pembangunan model klasifikasi sentimen untuk judul berita ekonomi CNBC Indonesia. Setelah data dipahami pada tahap EDA, langkah berikutnya adalah mengubah teks menjadi fitur numerik dan melatih model machine learning.

Teknik AI yang digunakan pada tahap ini adalah **Natural Language Processing (NLP)** dengan **TF-IDF** sebagai metode ekstraksi fitur, lalu **Support Vector Machine (SVM)** sebagai model klasifikasi. Target model adalah memprediksi sentimen judul berita menjadi `negatif`, `netral`, atau `positif`.


## 1. Load Dataset

Pada tahap ini, dataset dibaca kembali agar dapat digunakan untuk proses modeling. Library yang digunakan meliputi `pandas` untuk membaca data, `re` untuk membersihkan teks, `TfidfVectorizer` untuk mengubah teks menjadi angka, dan `SVC` untuk model SVM.

Path dataset dibuat relatif terhadap folder project. Tujuannya agar notebook tetap bisa dijalankan di komputer lain tanpa bergantung pada path lokal milik satu pengguna.


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import joblib
from pathlib import Path

data_path = Path("data/Dataset-CNBCI-Sentimented.csv")
if not data_path.exists():
    data_path = Path("../data/Dataset-CNBCI-Sentimented.csv")

df = pd.read_csv(data_path)
print("Data berhasil dimuat!")


Data berhasil dimuat!


### Interpretasi Hasil

Output menunjukkan bahwa dataset berhasil dimuat. Berdasarkan struktur dataset, data memiliki kolom utama `judul`, `tanggal`, dan `sentimen`.

Kolom `judul` akan menjadi input model, sedangkan kolom `sentimen` menjadi label atau target prediksi. Karena label sudah tersedia, pendekatan yang digunakan adalah supervised learning.


## 2. Preprocessing Teks

Sebelum teks masuk ke model, judul berita perlu dibersihkan terlebih dahulu. Fungsi `clean_text` melakukan beberapa langkah sederhana:

- Mengubah huruf menjadi lowercase.
- Menghapus link, mention, dan hashtag.
- Menghapus angka, tanda baca, dan simbol.
- Merapikan spasi berlebih.

Tahap ini penting karena model machine learning lebih mudah mempelajari pola teks yang bersih dan konsisten.


In [2]:
def clean_text(text):
    text = str(text).lower() 
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
    text = re.sub(r'\@\w+|\#','', text) 
    text = re.sub(r'[^a-zA-Z\s]', '', text) 
    text = re.sub(r'\s+', ' ', text).strip() 
    return text

df['cleaned_text'] = df['judul'].apply(clean_text)
print("Teks berhasil dibersihkan!")


Teks berhasil dibersihkan!


### Interpretasi Hasil

Output menunjukkan bahwa proses pembersihan teks berhasil dijalankan. Pada tahap ini, kolom baru `cleaned_text` dibuat sebagai versi bersih dari kolom `judul`.

Hasil preprocessing ini akan digunakan sebagai input untuk proses TF-IDF. Dengan teks yang lebih rapi, fitur kata yang dibentuk menjadi lebih konsisten.


## 3. Split Data dan Ekstraksi Fitur TF-IDF

Data dibagi menjadi data training dan data testing dengan rasio 80:20. Data training digunakan untuk melatih model, sedangkan data testing digunakan untuk mengevaluasi performa model pada data yang belum pernah dilihat.

Parameter `stratify=y` digunakan agar proporsi kelas sentimen pada data training dan testing tetap seimbang. Setelah itu, teks diubah menjadi fitur numerik menggunakan TF-IDF dengan maksimal 5.000 fitur dan kombinasi unigram-bigram.


In [3]:
X = df['cleaned_text']
y = df['sentimen'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)) 
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Dimensi data training: {X_train_vec.shape}")


Dimensi data training: (7855, 5000)


### Interpretasi Hasil

Output menunjukkan bahwa data training setelah TF-IDF memiliki bentuk `(7855, 5000)`. Artinya, terdapat 7.855 data training dan 5.000 fitur teks yang digunakan model. Dengan rasio testing 20%, data testing berjumlah 1.964 baris.

Angka 5.000 berarti model menggunakan maksimal 5.000 fitur kata atau pasangan kata yang dianggap penting. Dengan bentuk ini, teks sudah berubah menjadi format numerik yang dapat diproses oleh model SVM.


## 4. Training Model SVM dan Evaluasi

Model yang digunakan adalah Support Vector Machine dengan kernel linear. SVM cocok untuk klasifikasi teks karena hasil TF-IDF biasanya memiliki banyak fitur, dan SVM cukup kuat untuk mencari batas pemisah antar kelas pada data berdimensi besar.

Evaluasi dilakukan menggunakan metrik Accuracy, Precision, Recall, dan F1-Score. F1-Score penting karena menggabungkan precision dan recall sehingga lebih seimbang untuk menilai performa klasifikasi.


In [4]:
svm_model = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
print("Melatih model SVM...")
svm_model.fit(X_train_vec, y_train)

y_pred = svm_model.predict(X_test_vec)
print("\n=== Laporan Evaluasi Model ===")
print(classification_report(y_test, y_pred))

print(f"F1-Score : {f1_score(y_test, y_pred, average='weighted'):.2f}")


Melatih model SVM...

=== Laporan Evaluasi Model ===
              precision    recall  f1-score   support

     negatif       0.82      0.75      0.79       515
      netral       0.82      0.87      0.84       871
     positif       0.83      0.81      0.82       578

    accuracy                           0.82      1964
   macro avg       0.82      0.81      0.82      1964
weighted avg       0.82      0.82      0.82      1964

F1-Score : 0.82


### Interpretasi Hasil

Hasil evaluasi menunjukkan accuracy sebesar **0,82** atau sekitar **82%**. Nilai weighted average precision, recall, dan F1-Score juga berada di sekitar **0,82**.

Per kelas, model mendapatkan F1-Score sekitar 0,79 untuk `negatif`, 0,84 untuk `netral`, dan 0,82 untuk `positif`. Kelas netral menjadi yang paling baik karena jumlah datanya paling banyak. Kelas negatif sedikit lebih rendah, sehingga pada pengembangan berikutnya model dapat diperbaiki dengan preprocessing tambahan, penyesuaian parameter, atau penambahan data.


## 5. Export Model dan Vectorizer

Setelah model selesai dilatih, model SVM dan TF-IDF vectorizer disimpan ke folder `models`. File model dibutuhkan agar sistem Streamlit atau API dapat langsung melakukan prediksi tanpa melatih ulang dari awal.

Dua file yang disimpan adalah:

- `svm_sentiment_model.pkl`: model klasifikasi sentimen.
- `tfidf_vectorizer.pkl`: vectorizer untuk mengubah teks menjadi fitur TF-IDF.


In [5]:
models_dir = Path("models")
if not models_dir.exists():
    models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "svm_sentiment_model.pkl"
vectorizer_path = models_dir / "tfidf_vectorizer.pkl"

joblib.dump(svm_model, model_path)
joblib.dump(vectorizer, vectorizer_path)

print("Model dan Vectorizer berhasil diexport ke folder 'models'!")


Model dan Vectorizer berhasil diexport ke folder 'models'!


### Interpretasi Hasil

Output menunjukkan bahwa model dan vectorizer berhasil diekspor. Ini berarti tahap training sudah menghasilkan artefak model yang dapat digunakan kembali.

Langkah ini penting untuk deployment karena aplikasi hanya perlu memuat file `.pkl`, menerima input judul berita, membersihkan teks, melakukan transformasi TF-IDF, lalu menghasilkan prediksi sentimen.


## 6. Sanity Check Prediksi Model

Sanity check dilakukan untuk memastikan model yang sudah disimpan dapat dimuat kembali dan digunakan untuk prediksi sederhana. Pada tahap ini, tiga contoh berita dibuat secara manual untuk mewakili sentimen negatif, positif, dan netral.

Selain label prediksi, model juga menampilkan confidence score. Nilai confidence membantu melihat seberapa yakin model terhadap hasil prediksinya.


In [ ]:
models_dir = Path("models")
if not models_dir.exists():
    models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "svm_sentiment_model.pkl"
vectorizer_path = models_dir / "tfidf_vectorizer.pkl"

loaded_model = joblib.load(model_path)
loaded_vectorizer = joblib.load(vectorizer_path)

berita_baru = [
    "IHSG anjlok parah setelah pengumuman kenaikan suku bunga The Fed yang agresif.", 
    "Investasi asing masuk triliunan rupiah, ekonomi Indonesia meroket pesat!", 
    "Pemerintah merilis laporan anggaran kuartal ketiga pada hari ini." 
]

berita_bersih = [clean_text(berita) for berita in berita_baru]

fitur_baru = loaded_vectorizer.transform(berita_bersih)
prediksi = loaded_model.predict(fitur_baru)
probabilitas = loaded_model.predict_proba(fitur_baru) 

print("=== Hasil Sanity Check Model ===")
for i in range(len(berita_baru)):
    print(f"\nBerita Asli   : {berita_baru[i]}")
    print(f"Hasil Prediksi: {prediksi[i].upper()}")
    print(f"Confidence    : {max(probabilitas[i]) * 100:.2f}%")


=== Hasil Sanity Check Model ===

Berita Asli   : IHSG anjlok parah setelah pengumuman kenaikan suku bunga The Fed yang agresif.
Hasil Prediksi: NEGATIF
Confidence    : 93.73%

Berita Asli   : Investasi asing masuk triliunan rupiah, ekonomi Indonesia meroket pesat!
Hasil Prediksi: POSITIF
Confidence    : 92.25%

Berita Asli   : Pemerintah merilis laporan anggaran kuartal ketiga pada hari ini.
Hasil Prediksi: NETRAL
Confidence    : 76.61%


### Interpretasi Hasil

Hasil sanity check menunjukkan bahwa model dapat memprediksi tiga contoh berita dengan sesuai:

- Berita tentang IHSG anjlok diprediksi `negatif` dengan confidence 93,73%.
- Berita tentang investasi asing dan ekonomi meroket diprediksi `positif` dengan confidence 92,25%.
- Berita laporan anggaran diprediksi `netral` dengan confidence 76,61%.

Confidence untuk kelas netral lebih rendah dibanding dua contoh lain. Ini wajar karena kalimat netral sering lebih sulit dibedakan dari berita positif atau negatif jika ada kata ekonomi yang konteksnya tidak terlalu kuat. Sanity check ini bukan pengganti evaluasi utama, tetapi berguna untuk memastikan pipeline model berjalan dari input sampai output.


## Ringkasan Modeling

Notebook ini berhasil membangun pipeline NLP sederhana untuk klasifikasi sentimen berita ekonomi. Tahap yang dilakukan meliputi load dataset, preprocessing teks, split data, ekstraksi fitur TF-IDF, training model SVM, evaluasi, export model, dan sanity check.

Hasil evaluasi model cukup baik dengan accuracy dan weighted F1-Score sekitar **82%**. Model sudah dapat membedakan sentimen negatif, netral, dan positif pada judul berita. Namun, performa masih dapat ditingkatkan, terutama pada kelas negatif yang memiliki F1-Score paling rendah dibanding kelas lain.

Untuk kebutuhan UAS, notebook ini mendukung bagian **The Tech** dan **The Value** pada artikel Medium. Model yang sudah diekspor juga dapat digunakan pada aplikasi Streamlit sebagai bagian dari sistem AI end-to-end.
